In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

!pip install lightgbm shap -q
import lightgbm as lgb
import shap

print("✓ Ready")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data_path = "/content/drive/MyDrive/AI&MED_Final/data/cleaned/"
out_path  = "/content/drive/MyDrive/AI&MED_Final/"

df = pd.read_csv(data_path + "nhanes_clean.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"MetS prevalence: {df['MetS'].mean()*100:.1f}%")
df.head()

In [ ]:
# Baseline: BMI, age, sex
X_base = df[["bmi", "age", "sex"]].copy()

# Enhanced: + WHtR, BMI×sex, race/ethnicity dummies
X_enh = df[["bmi", "age", "sex"]].copy()
X_enh["whtr"] = df["waist_cm"] / df["height_cm"]
X_enh["bmi_x_sex"] = df["bmi"] * df["sex"]

race_dummies = pd.get_dummies(df["race_eth"], prefix="race", drop_first=True)
X_enh = pd.concat([X_enh, race_dummies], axis=1)

y = df["MetS"]

print(f"Baseline features ({X_base.shape[1]}): {X_base.columns.tolist()}")
print(f"Enhanced features ({X_enh.shape[1]}): {X_enh.columns.tolist()}")
print(f"Target: {y.value_counts().to_dict()}")

In [ ]:
def get_models():
    return {
        "Elastic Net LR": LogisticRegression(
            penalty="elasticnet", solver="saga", l1_ratio=0.5,
            max_iter=5000, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_leaf=5,
            random_state=42, n_jobs=-1
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            num_leaves=31, random_state=42, verbose=-1
        ),
    }

def evaluate(X, y, feature_set_name):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    for model_name, model in get_models().items():
        aucs, precs, recs, f1s, specs = [], [], [], [], []
        fold_y_true, fold_y_prob = [], []

        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            if "LR" in model_name:
                scaler = StandardScaler()
                X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
                X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

            model.fit(X_train, y_train)
            y_prob = model.predict_proba(X_test)[:, 1]
            y_pred = (y_prob >= 0.5).astype(int)

            aucs.append(roc_auc_score(y_test, y_prob))
            precs.append(precision_score(y_test, y_pred, zero_division=0))
            recs.append(recall_score(y_test, y_pred))
            f1s.append(f1_score(y_test, y_pred))
            tn = ((y_pred == 0) & (y_test == 0)).sum()
            fp = ((y_pred == 1) & (y_test == 0)).sum()
            specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

            fold_y_true.extend(y_test)
            fold_y_prob.extend(y_prob)

        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "AUROC": np.mean(aucs),
            "Precision": np.mean(precs),
            "Recall": np.mean(recs),
            "Specificity": np.mean(specs),
            "F1": np.mean(f1s),
            "y_true": fold_y_true,
            "y_prob": fold_y_prob,
        })
        print(f"  {model_name:20s} | AUROC: {np.mean(aucs):.4f} | F1: {np.mean(f1s):.4f}")

    return results

print("✓ Evaluation function ready")

In [ ]:
print("=" * 60)
print("BASELINE (BMI, age, sex)")
print("=" * 60)
results_base = evaluate(X_base, y, "Baseline")

print(f"\n{'=' * 60}")
print("ENHANCED (+ WHtR, BMI×sex, race/ethnicity)")
print("=" * 60)
results_enh = evaluate(X_enh, y, "Enhanced")

all_results = results_base + results_enh

In [ ]:
summary = pd.DataFrame(all_results).drop(columns=["y_true", "y_prob"])
summary = summary.round(4)
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = ["Elastic Net LR", "Random Forest", "LightGBM"]

for i, mname in enumerate(model_names):
    ax = axes[i]
    for res in all_results:
        if res["Model"] == mname:
            fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
            label = f"{res['Feature Set']} (AUC={res['AUROC']:.3f})"
            ax.plot(fpr, tpr, label=label, linewidth=2)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(mname, fontsize=14)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    ax.set_aspect("equal")

plt.suptitle("ROC Curves: Baseline vs Enhanced Feature Set", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(out_path + "roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved roc_curves.png")

In [ ]:
best = max(all_results, key=lambda x: x["AUROC"])
print(f"Best model: {best['Model']} ({best['Feature Set']}) — AUROC: {best['AUROC']:.4f}")

if best["Feature Set"] == "Baseline":
    X_shap = X_base.copy()
else:
    X_shap = X_enh.copy()

if "LightGBM" in best["Model"]:
    shap_model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        num_leaves=31, random_state=42, verbose=-1
    )
elif "Random Forest" in best["Model"]:
    shap_model = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    )
else:
    scaler = StandardScaler()
    X_shap = pd.DataFrame(scaler.fit_transform(X_shap), columns=X_shap.columns)
    shap_model = LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5,
        max_iter=5000, random_state=42
    )

shap_model.fit(X_shap, y)

if "LightGBM" in best["Model"] or "Random Forest" in best["Model"]:
    explainer = shap.TreeExplainer(shap_model)
else:
    explainer = shap.LinearExplainer(shap_model, X_shap)

shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

print("✓ SHAP values computed")

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title(f"SHAP Feature Importance — {best['Model']} ({best['Feature Set']})", fontsize=14)
plt.tight_layout()
plt.savefig(out_path + "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved shap_summary.png")

In [ ]:
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.title(f"Mean |SHAP| — {best['Model']} ({best['Feature Set']})", fontsize=14)
plt.tight_layout()
plt.savefig(out_path + "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved shap_bar.png")

In [ ]:
summary.to_csv(out_path + "model_results.csv", index=False)
print("✓ Saved model_results.csv")
print("\nAll outputs saved to AI&MED_Final/:")
print("  - model_results.csv")
print("  - roc_curves.png")
print("  - shap_summary.png")
print("  - shap_bar.png")

In [ ]:
def evaluate_with_ci(X, y, feature_set_name):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    for model_name, model in get_models().items():
        aucs, precs, recs, f1s, specs = [], [], [], [], []

        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            if "LR" in model_name:
                scaler = StandardScaler()
                X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
                X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

            model.fit(X_train, y_train)
            y_prob = model.predict_proba(X_test)[:, 1]
            y_pred = (y_prob >= 0.5).astype(int)

            aucs.append(roc_auc_score(y_test, y_prob))
            precs.append(precision_score(y_test, y_pred, zero_division=0))
            recs.append(recall_score(y_test, y_pred))
            f1s.append(f1_score(y_test, y_pred))
            tn = ((y_pred == 0) & (y_test == 0)).sum()
            fp = ((y_pred == 1) & (y_test == 0)).sum()
            specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "AUROC": f"{np.mean(aucs):.4f} ± {np.std(aucs):.4f}",
            "Precision": f"{np.mean(precs):.4f} ± {np.std(precs):.4f}",
            "Recall": f"{np.mean(recs):.4f} ± {np.std(recs):.4f}",
            "Specificity": f"{np.mean(specs):.4f} ± {np.std(specs):.4f}",
            "F1": f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
        })

    return results

ci_base = evaluate_with_ci(X_base, y, "Baseline")
ci_enh  = evaluate_with_ci(X_enh, y, "Enhanced")

ci_df = pd.DataFrame(ci_base + ci_enh)
print(ci_df.to_string(index=False))
ci_df.to_csv(out_path + "model_results_with_ci.csv", index=False)
print("\n✓ Saved model_results_with_ci.csv")

In [ ]:
# DeLong test implementation
from scipy import stats

def delong_roc_test(y_true, y_prob1, y_prob2):
    """Bootstrapped comparison of two AUROCs."""
    n_boot = 10000
    rng = np.random.RandomState(42)
    y_true = np.array(y_true)
    y_prob1 = np.array(y_prob1)
    y_prob2 = np.array(y_prob2)
    n = len(y_true)

    auc1 = roc_auc_score(y_true, y_prob1)
    auc2 = roc_auc_score(y_true, y_prob2)
    observed_diff = auc2 - auc1

    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        try:
            a1 = roc_auc_score(y_true[idx], y_prob1[idx])
            a2 = roc_auc_score(y_true[idx], y_prob2[idx])
            diffs.append(a2 - a1)
        except ValueError:
            continue

    diffs = np.array(diffs)
    se = np.std(diffs)
    z = observed_diff / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    return observed_diff, z, p_value

print("Paired Bootstrap Test: Enhanced vs Baseline AUROC")
print("=" * 65)

model_names = ["Elastic Net LR", "Random Forest", "LightGBM"]
for mname in model_names:
    base_res = [r for r in all_results if r["Model"] == mname and r["Feature Set"] == "Baseline"][0]
    enh_res  = [r for r in all_results if r["Model"] == mname and r["Feature Set"] == "Enhanced"][0]

    diff, z, p = delong_roc_test(base_res["y_true"], base_res["y_prob"], enh_res["y_prob"])

    sig = "✓ Significant" if p < 0.05 else "✗ Not significant"
    print(f"  {mname:20s} | ΔAUC: {diff:+.4f} | z: {z:.3f} | p: {p:.4f} | {sig}")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, mname in enumerate(model_names):
    ax = axes[i]
    for res in all_results:
        if res["Model"] == mname:
            prec_arr, rec_arr, _ = precision_recall_curve(res["y_true"], res["y_prob"])
            ap = average_precision_score(res["y_true"], res["y_prob"])
            label = f"{res['Feature Set']} (AP={ap:.3f})"
            ax.plot(rec_arr, prec_arr, label=label, linewidth=2)

    ax.set_title(mname, fontsize=14)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="upper right")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.suptitle("Precision-Recall Curves: Baseline vs Enhanced", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(out_path + "pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved pr_curves.png")

In [ ]:
best_enh = [r for r in all_results if r["Feature Set"] == "Enhanced"]
best_enh = max(best_enh, key=lambda x: x["AUROC"])

y_true_arr = np.array(best_enh["y_true"])
y_prob_arr = np.array(best_enh["y_prob"])

thresholds = np.arange(0.1, 0.9, 0.05)
rows = []
for t in thresholds:
    y_pred_t = (y_prob_arr >= t).astype(int)
    rec  = recall_score(y_true_arr, y_pred_t)
    prec = precision_score(y_true_arr, y_pred_t, zero_division=0)
    f1_t = f1_score(y_true_arr, y_pred_t)
    tn = ((y_pred_t == 0) & (y_true_arr == 0)).sum()
    fp = ((y_pred_t == 1) & (y_true_arr == 0)).sum()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    rows.append({"Threshold": t, "Recall": rec, "Precision": prec, "F1": f1_t, "Specificity": spec})

thresh_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df["Threshold"], thresh_df["Recall"], "o-", label="Recall (Sensitivity)", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["Precision"], "s-", label="Precision", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["F1"], "^-", label="F1 Score", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["Specificity"], "d-", label="Specificity", linewidth=2)
ax.set_xlabel("Classification Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(f"Threshold Analysis — {best_enh['Model']} (Enhanced)", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nThreshold Table:")
print(thresh_df.round(4).to_string(index=False))
print("\n✓ Saved threshold_analysis.png")

In [ ]:
print("SUBGROUP ANALYSIS BY SEX")
print("=" * 70)

# Use best enhanced model — retrain and predict per fold, store with sex info
best_model_name = max(all_results, key=lambda x: x["AUROC"])["Model"]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true, all_y_prob, all_sex = [], [], []

for train_idx, test_idx in skf.split(X_enh, y):
    X_train, X_test = X_enh.iloc[train_idx], X_enh.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    sex_test = df["sex"].iloc[test_idx]

    if "LR" in best_model_name:
        scaler = StandardScaler()
        X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_enh.columns)
        X_test_s  = pd.DataFrame(scaler.transform(X_test), columns=X_enh.columns)
    else:
        X_train_s, X_test_s = X_train, X_test

    model = get_models()[best_model_name]
    model.fit(X_train_s, y_train)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    all_y_true.extend(y_test.values)
    all_y_prob.extend(y_prob)
    all_sex.extend(sex_test.values)

all_y_true = np.array(all_y_true)
all_y_prob = np.array(all_y_prob)
all_sex = np.array(all_sex)

for sex_val, sex_label in [(1, "Male"), (0, "Female")]:
    mask = all_sex == sex_val
    auc = roc_auc_score(all_y_true[mask], all_y_prob[mask])
    y_pred = (all_y_prob[mask] >= 0.5).astype(int)
    rec = recall_score(all_y_true[mask], y_pred)
    prec = precision_score(all_y_true[mask], y_pred, zero_division=0)
    f1_v = f1_score(all_y_true[mask], y_pred)
    prev = all_y_true[mask].mean()
    print(f"  {sex_label:8s} | n={mask.sum():,} | MetS prev: {prev:.1%} | AUROC: {auc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1_v:.4f}")

In [ ]:
print("\nSUBGROUP ANALYSIS BY RACE/ETHNICITY")
print("=" * 70)

all_race = df["race_eth"].values

# Use same fold predictions from Cell 16
all_race_fold = []
for train_idx, test_idx in skf.split(X_enh, y):
    all_race_fold.extend(df["race_eth"].iloc[test_idx].values)
all_race_fold = np.array(all_race_fold)

# NHANES RIDRETH3 codes
race_labels = {
    1.0: "Mexican American",
    2.0: "Other Hispanic",
    3.0: "Non-Hispanic White",
    4.0: "Non-Hispanic Black",
    6.0: "Non-Hispanic Asian",
    7.0: "Other/Multi-Racial",
}

rows = []
for code, label in sorted(race_labels.items()):
    mask = all_race_fold == code
    if mask.sum() < 20:
        continue
    try:
        auc = roc_auc_score(all_y_true[mask], all_y_prob[mask])
    except ValueError:
        auc = float("nan")
    y_pred = (all_y_prob[mask] >= 0.5).astype(int)
    rec = recall_score(all_y_true[mask], y_pred, zero_division=0)
    prec = precision_score(all_y_true[mask], y_pred, zero_division=0)
    f1_v = f1_score(all_y_true[mask], y_pred, zero_division=0)
    prev = all_y_true[mask].mean()
    rows.append({"Group": label, "n": mask.sum(), "MetS Prev": f"{prev:.1%}",
                 "AUROC": round(auc, 4), "Precision": round(prec, 4),
                 "Recall": round(rec, 4), "F1": round(f1_v, 4)})
    print(f"  {label:22s} | n={mask.sum():>4} | MetS: {prev:.1%} | AUROC: {auc:.4f} | Rec: {rec:.4f} | F1: {f1_v:.4f}")

subgroup_df = pd.DataFrame(rows)
subgroup_df.to_csv(out_path + "subgroup_results.csv", index=False)
print("\n✓ Saved subgroup_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- By sex ---
ax = axes[0]
sex_labels, sex_aucs = [], []
for sex_val, sex_label in [(1, "Male"), (0, "Female")]:
    mask = all_sex == sex_val
    auc = roc_auc_score(all_y_true[mask], all_y_prob[mask])
    sex_labels.append(sex_label)
    sex_aucs.append(auc)
ax.barh(sex_labels, sex_aucs, color=["#4A90D9", "#D94A7A"], height=0.5)
ax.set_xlim(0.7, 0.9)
ax.set_xlabel("AUROC")
ax.set_title("Enhanced Model AUROC by Sex", fontsize=13)
for i, v in enumerate(sex_aucs):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=12)

# --- By race/ethnicity ---
ax = axes[1]
race_names, race_aucs = [], []
for code, label in sorted(race_labels.items()):
    mask = all_race_fold == code
    if mask.sum() < 20:
        continue
    try:
        auc = roc_auc_score(all_y_true[mask], all_y_prob[mask])
        race_names.append(label)
        race_aucs.append(auc)
    except ValueError:
        continue
ax.barh(race_names, race_aucs, color="#4A90D9", height=0.5)
ax.set_xlim(0.65, 0.95)
ax.set_xlabel("AUROC")
ax.set_title("Enhanced Model AUROC by Race/Ethnicity", fontsize=13)
for i, v in enumerate(race_aucs):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=11)

plt.tight_layout()
plt.savefig(out_path + "subgroup_auroc.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved subgroup_auroc.png")

In [ ]:
import seaborn as sns

# Reuse X_shap and shap_values already computed above
shap_df = pd.DataFrame(shap_values, columns=X_shap.columns)
shap_df["race"] = df["race_eth"].values

race_labels = {
    1.0: "Mexican American", 2.0: "Other Hispanic",
    3.0: "NH White", 4.0: "NH Black",
    6.0: "NH Asian", 7.0: "Other/Multi"
}

mean_abs = {}
for code, label in race_labels.items():
    mask = shap_df["race"] == code
    if mask.sum() < 20: continue
    mean_abs[label] = shap_df.loc[mask, X_shap.columns].abs().mean()

heatmap_df = pd.DataFrame(mean_abs).T

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Mean |SHAP|"})
ax.set_title("Mean |SHAP| by Race/Ethnicity — which features drive predictions per group?", pad=10)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig(out_path + "shap_heatmap_by_race.png", dpi=150, bbox_inches="tight")
plt.show()
print(heatmap_df.round(4).to_string())

In [ ]:
y_pred_full = (shap_model.predict_proba(X_shap)[:, 1] >= 0.5).astype(int)
race_arr = df["race_eth"].values

rows = []
for code, label in race_labels.items():
    mask = (race_arr == code) & (y.values == 1)  # MetS-positive only
    if mask.sum() < 5: continue
    miss_rate = (y_pred_full[mask] == 0).mean()
    rows.append({"Group": label, "MetS+ n": mask.sum(),
                 "Recall": round(1 - miss_rate, 3),
                 "Miss rate": round(miss_rate, 3)})

fn_df = pd.DataFrame(rows).sort_values("Miss rate", ascending=False)
print(fn_df.to_string(index=False))
fn_df.to_csv(out_path + "miss_rate_by_race.csv", index=False)

In [ ]:
whtr_raw = df["waist_cm"] / df["height_cm"]

data, labels = [], []
for code, label in race_labels.items():
    mask = (race_arr == code) & (y.values == 1)
    if mask.sum() < 10: continue
    data.append(whtr_raw[mask].values)
    labels.append(f"{label} (n={mask.sum()})")

fig, ax = plt.subplots(figsize=(10, 5))
ax.violinplot(data, vert=False, showmedians=True)
ax.set_yticks(range(1, len(labels) + 1))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0.5, color="red", linestyle="--", alpha=0.7, label="WHtR=0.5 threshold")
ax.set_xlabel("WHtR (among MetS-positive participants only)")
ax.set_title("WHtR Distribution Among MetS+ Participants\n"
             "Lower median for NH Asian = WHtR is a weaker risk signal for that group")
ax.legend()
plt.tight_layout()
plt.savefig(out_path + "whtr_dist_mets_positive.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nMedian WHtR among MetS+ by race:")
for code, label in race_labels.items():
    mask = (race_arr == code) & (y.values == 1)
    if mask.sum() < 10: continue
    print(f"  {label:22s}: {whtr_raw[mask].median():.3f}  (n={mask.sum()})")

In [ ]:
print("All outputs saved to AI&MED_Final/:")
print("  - model_results_with_ci.csv")
print("  - subgroup_results.csv")
print("  - roc_curves.png")
print("  - pr_curves.png")
print("  - threshold_analysis.png")
print("  - shap_summary.png")
print("  - shap_bar.png")
print("  - subgroup_auroc.png")
print("  - shap_heatmap_by_race.png")
print("  - miss_rate_by_race.csv")
print("  - whtr_dist_mets_positive.png")
print("\n✓ Done!")